In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


 Because of this is notebook we don't run from Makfile directory so use " getcwd() "

In [26]:
current_dir = Path(os.getcwd())
print(f"the path is {current_dir}")
file = current_dir / '../data/PRSA_Data_Aotizhongxin_20130301-20170228.csv'

the path is /home/ahmed-sulieman/Desktop/AI_projects/Air_Pol/notebook


# Makefile

run this below code when you run from the Makfile

In [27]:
# current_dir = Path(__file__).parent
# file = current_dir / '../archive/PRSA_Data_Aotizhongxin_20130301-20170228.csv'

In [28]:
if not file.exists():
    print('file not exist')
    raise FileNotFoundError(f"File {file} does not exist.")

In [29]:
df = pd.read_csv(file)

In [30]:
df = df.drop(columns=['No', 'station'])
df_b = df.copy()

In [31]:
df.fillna(method='ffill', inplace=True)
df_b.dropna(inplace=True)

/tmp/ipykernel_82348/490933577.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [32]:
df.head(5)

,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0


In [ ]:
def split_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Split the 'date' column into separate 'year', 'month', 'day', and 'hour' columns.
    Parameters:
    df (pd.DataFrame): The input DataFrame containing a 'date' column.
    Returns:
        pd.DataFrame: A DataFrame with the 'date' column split into separate columns.
    """
    try:
        train_raw = df[df['year'] < 2016].copy()
        test_raw = df[df['year'] >= 2016].copy()
        df_test = test_raw.dropna().copy()
        logging.info("Date column split successfully.")
        return train_raw, df_test
    except Exception as e:
        logging.error(f"Error occurred while splitting date: {e}")
        return df

def get_processed_data(file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load and process the air pollution data from a CSV file.
    Parameters:
    file (str): The name of the CSV file containing the data.
    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: A tuple containing the processed DataFrame 
    with filled missing values and a copy of the DataFrame with dropped missing values.
    """
    try:
        current_dir = Path(os.getcwd())
        file_path = current_dir / '../data' / file
        # current_dir = Path(__file__).parent
        # file_path = current_dir / '../archive' / file

        if not file_path.exists():
            logging.error(f"File {file_path} does not exist.")
            print('file not exist')
            return None, None, None
        df = pd.read_csv(file_path)
        df = df.drop(columns=['No', 'station'])
        df = pd.get_dummies(df, columns=['wd'])

        df_train, df_test = split_dataset(df)
        
        if df_train is None or df_test is None:
            logging.error("Failed to split dataset.")
            return None, None, None
        
        df_dropped_values = df_train.copy()

        df_train.fillna(method='ffill', inplace=True)
        df_dropped_values.dropna(inplace=True)

        logging.info("Data loaded and processed successfully.")
        return df_train, df_dropped_values, df_test
    except Exception as e:
        logging.error(f"Error occurred: {e}")
        return None, None, None

    

In [34]:
file = 'PRSA_Data_Aotizhongxin_20130301-20170228.csv'
df, df_b, df_test = get_processed_data(file)
print(f"df shape: {df.shape} | df_b shape: {df_b.shape} | df_test shape: {df_test.shape}")

2026-05-13 09:01:48,149 - INFO - Date column split successfully.
/tmp/ipykernel_82348/3601428766.py:51: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_train.fillna(method='ffill', inplace=True)
2026-05-13 09:01:48,174 - INFO - Data loaded and processed successfully.


df shape: (24864, 31) | df_b shape: (22240, 31) | df_test shape: (9636, 31)


In [35]:
df.head(5)

,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,...,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,...,True,False,False,False,False,False,False,False,False,False
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,...,False,False,False,False,False,False,False,False,False,False
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,...,True,False,False,False,False,False,False,False,False,False
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,...,False,True,False,False,False,False,False,False,False,False
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,...,False,False,False,False,False,False,False,False,False,False
